# Local Corp Income Agents Notebook

In [ ]:
from __future__ import annotations

import json
import os
import sys

from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv


PROJECT_ROOT = Path().resolve()
# override=False: a value already set (e.g. from dbutils.secrets in the next
# cell) is never replaced by an empty line in a stray .env file.
load_dotenv(PROJECT_ROOT / ".env", override=False)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# LLM settings — pick ONE block, uncomment it, fill the <...>.
# A file named exactly ".env" next to this notebook works too (cell 2 reads it);
# ".env.example" is only a template and is never read.
# Values set here win over .env. Run this cell AFTER any restartPython().
import os

# --- A. OpenAI (key cá nhân / công ty) -------------------------------------
# os.environ["LLM_PROVIDER"] = "openai"
# os.environ["OPENAI_API_KEY"] = dbutils.secrets.get("<scope>", "<key-name>")   # or "sk-..." for a quick test
# os.environ["MODEL_DOCUMENT"] = "gpt-4o-mini"
# os.environ["MODEL_ECONOMY"] = "gpt-4o-mini"
# os.environ["MODEL_PREMIUM"] = "gpt-4o"

# --- B. Databricks model serving (no key of your own; data stays in the workspace)
# _ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
# os.environ["LLM_PROVIDER"] = "databricks"
# os.environ["DATABRICKS_HOST"] = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
# os.environ["DATABRICKS_TOKEN"] = _ctx.apiToken().get()
# os.environ["MODEL_DOCUMENT"] = os.environ["MODEL_ECONOMY"] = os.environ["MODEL_PREMIUM"] = "<endpoint name from Serving>"

# --- C. Anthropic ------------------------------------------------------------
# os.environ["LLM_PROVIDER"] = "anthropic"
# os.environ["ANTHROPIC_API_KEY"] = dbutils.secrets.get("<scope>", "<key-name>")
# os.environ["MODEL_DOCUMENT"] = os.environ["MODEL_ECONOMY"] = "<model>"
# os.environ["MODEL_PREMIUM"] = "<model>"

# OCR: Vietnamese only reads diacritics far better than vie+eng
os.environ.setdefault("OCR_LANG", "vie")
# os.environ["OCR_CACHE_DIR"] = "/Volumes/<catalog>/<schema>/<volume>/ocr_cache"


In [ ]:
# [12/11] Chạy NHIỀU case một lần (cell "Run" cuối notebook).
# Mỗi case = một thư mục trong CASES_DIR, bên trong có các thư mục hộp hồ sơ:
# ho_so_phap_ly, ho_so_tai_chinh, ho_so_vay_von, ho_so_tai_san_dam_bao,
# ho_so_noi_bo, ho_so_soan_thao_noi_bo, ho_so_khac
# (CSV/MD/PDF/TXT/XLS/XLSX/XML/DOCX/JPG/PNG).
CASES_DIR = PROJECT_ROOT / "samples"            # Databricks: Path("/Volumes/<catalog>/<schema>/<volume>/ho_so")
TESTCASE_IDS = ["case_1", "case_2", "case_3"]   # 1 case cũng được: ["case_1"]
# Tất cả thư mục trong CASES_DIR:
# TESTCASE_IDS = sorted(p.name for p in Path(CASES_DIR).iterdir() if p.is_dir())

# Thiết lập riêng từng case (nếu có) — ghi đè thiết lập chung bên dưới cho đúng case đó.
# "input_paths": case nằm ngoài CASES_DIR (hoặc ghép nhiều thư mục cho 1 case).
PER_CASE = {
    # "case_2": {"borrower_names": ["Nguyễn Văn A"], "annual_lt_principal": 5_000_000_000},
    # "case_3": {"input_paths": ["/Volumes/<catalog>/<schema>/<volume>/khdn_xyz"]},
}

# ---- Thiết lập chung cho mọi case ----
# KHDN
LOAN_PROGRAM = "KHDN"

# Possible value: CORP_INCOME_AGENT
AGENT = "CORP_INCOME_AGENT"

# The 12-month checks end on this date (dd/mm/yyyy); None = today
LOAN_DATE = None
# KH cá nhân vay vốn. Để trống [] = tự tìm tên thành viên góp vốn / NĐDPL trong
# hồ sơ vay vốn (ho_so_vay_von, rồi ho_so_soan_thao_noi_bo). Điền tay chỉ khi muốn ghi đè.
BORROWER_NAMES = []

# Optional: BRD B2/B3/B4 — {item: JSON}, item = legal_basic | shareholders | legal_rep | liabilities
INTERNAL_DATA = {}
EXTERNAL_DATA = {}
WATCHLIST = []          # [{"name": "...", "tax_code": "...", "list": "AML"}]

# [BRD 16/10] Nghĩa vụ trả gốc TDH hằng năm của DN (đồng) — mục 6 "LNST sử dụng để trả
# nghĩa vụ TDH". None = chưa có (tính 0 và ghi chú trong báo cáo).
ANNUAL_LT_PRINCIPAL = None

OUTPUT_DIR = PROJECT_ROOT / "logs"
MAX_CHARS_PER_DOCUMENT = 120_000

In [ ]:
from src.config import Config, build_llm

config = Config(
    # Phân loại tài liệu (chỉ khi từ khóa không chắc): 0.0 để mỗi lần chạy cho cùng kết quả.
    document_llm=build_llm(
        "MODEL_DOCUMENT",
        temperature=0.0
    ),
    legal_basic_extraction_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.0,
    ),
    shareholders_extraction_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.0,
    ),
    legal_rep_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    liabilities_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    # Phụ lục 01 mục 3: số liệu BCTC (tỷ số tính bằng code, không do LLM)
    financial_extraction_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.0,
    ),
    # Viết các ô nhận xét / HĐKD / CIC / kết luận theo guidance
    # Văn phong: 0.2 cho câu chữ tự nhiên hơn mà kết luận vẫn ổn định giữa các lần chạy.
    # Không nên > 0.3 cho tờ trình tín dụng. Model mạnh hơn (Sonnet) cải thiện nhiều hơn temperature.
    report_writer_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.2,
    ),
    # [BRD 16/10] tờ khai VAT / CIC / phương án kinh doanh: để trống thì dùng
    # financial_extraction_llm. Muốn model riêng thì thêm, ví dụ:
    # cic_extraction_llm=build_llm("MODEL_PREMIUM", temperature=0.0),
    # business_model_extraction_llm=build_llm("MODEL_ECONOMY", temperature=0.0),  # [BRD 27/10] mục 2
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)

In [ ]:
# Check the models BEFORE the run: one tiny call per model.
# Stops "Run all" here with the real cause (key, URL, firewall, quota…)
# instead of failing every extraction after the OCR.
from src.llm_check import check_llm_setup

check_llm_setup(config)


In [ ]:
from src.types import (
    AgentName,
    WorkflowMode,
    DocumentAgentName,
    ClassifiedDocument,
    UnderwritingGraphState,
    to_dict_list,
    extract_text_from_agent_output,
    truncate_text,
)

In [ ]:
from src.agents.documents.document_classification import (
    document_type_scores,
    rule_classify_document,
)
from src.agents.documents.document_discovery import (
    compute_file_hash,
    discover_documents,
    resolve_input_path,
)
from src.utils.common import (
    SUPPORTED_EXTENSIONS,
    normalize_text,
)
from src.types import VALID_DOCUMENT_AGENTS
from src.agents.documents.document_matrix import (
    agent_relevance_for_type,
    load_matrix,
    primary_agent_for_type,
)

_matrix = load_matrix()
print(
    f"Document matrix v{_matrix.version}: {len(_matrix.types)} document types \n"
    f"Loan programs: {', '.join(_matrix.loan_programs)}"
)

In [ ]:
from src.agents.specialist import (
    CorpIncomeAnalysis,
)
from src.agents.supervisor import Supervisor

## Run — chạy nhiều case [12/11]

Thứ tự: chạy các cell trên (thông số → config → kiểm tra model → import) rồi chạy cell dưới.

- Các case chạy **lần lượt**; case lỗi không làm dừng các case khác.
- Mỗi case lưu vào `OUTPUT_DIR/<case>_<giờ>/`: `final_response_<case>.pdf / .md`, JSON, `corp_income.xlsx`, `ocr/`.
- Bảng tổng hợp hiện cuối cell và lưu `OUTPUT_DIR/batch_<giờ>.json`: status `OK` / `THIẾU TRÍCH XUẤT` / `LỖI`.

In [ ]:
from src.agents.supervisor import Supervisor
from src.utils.batch import run_cases
import pandas as pd

supervisor = Supervisor(config)
summary = run_cases(
    supervisor, TESTCASE_IDS,
    samples_dir=CASES_DIR,
    output_dir=OUTPUT_DIR,
    per_case=PER_CASE,
    agent=AGENT,
    loan_program=LOAN_PROGRAM,
    loan_date=LOAN_DATE,
    borrower_names=BORROWER_NAMES,
    internal_data=INTERNAL_DATA,
    external_data=EXTERNAL_DATA,
    watchlist=WATCHLIST,
    annual_lt_principal=ANNUAL_LT_PRINCIPAL,
)

display(pd.DataFrame(summary))

# Case không OK (vd 429 kéo dài) → bỏ # dòng dưới, chạy lại cell này chỉ cho các case đó:
# TESTCASE_IDS = [r["case"] for r in summary if r["status"] != "OK"]

## Xem kết quả một case (tuỳ chọn)

Đổi `CASE` thành case muốn xem: hiện báo cáo, các lượt trích xuất thất bại và kết luận từng mục.

In [ ]:
from IPython.display import Markdown, display

CASE = TESTCASE_IDS[0]

_row = next(r for r in summary if r["case"] == CASE)
if _row["status"] == "LỖI":
    print(f"{CASE}: LỖI — {_row['error']}")
else:
    _dir = Path(_row["output"])
    print(f"{CASE}: {_row['status']} — nhóm DN: {_row['corp_group']} — {_dir}")
    for _f in filter(None, _row["failed"].split("; ")):
        print("  - trích xuất thất bại:", _f)
    _corp = json.loads((_dir / "corp_income.json").read_text(encoding="utf-8")) or {}
    for _n, _item in (_corp.get("items") or {}).items():
        print(f"  {_n}. {_item['title']}: {_item['conclusion']}")
    display(Markdown((_dir / f"final_response_{CASE}.md").read_text(encoding="utf-8")))